# Agent LangChain avec ChatGroq

Ce notebook démontre la création d'un agent intelligent utilisant LangChain et ChatGroq avec des outils personnalisés.


In [47]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

import numexpr as ne

from dotenv import load_dotenv

load_dotenv()


True

## Définition des outils

Nous créons deux fonctions qui seront utilisées comme outils par l'agent. Les docstrings sont nécessaires car elles permettent au LLM d'avoir une définition des outils à sa disposition.

In [48]:
def calculer(expression: str):
    """Effectue un calcul mathématique."""
    return str(ne.evaluate(expression)) # A utiliser à la place de eval() qui est très dangereux 

def whois_seb():
    """Donne une description de Séb."""
    return str("Séb est un binoclard chauve. Il a une piste d'atterrisage pour mouche sur le crâne et un style vestimentaire douteux.")

calculer("2+2")


'4'

## Initialisation du modèle LLM


In [49]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.7)

## Création de l'agent

L'agent est configuré avec le modèle LLM, les outils et un prompt système :


In [50]:
agent = create_agent(
    model=llm,
    tools=[calculer, whois_seb],
    system_prompt="Tu es un assistant intelligent. Utilise les outils si nécessaire."
)

## Fonction pour poser une question

Cette fonction permet d'interroger l'agent et de récupérer sa réponse :


In [51]:
def ask(question: str) -> str:
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    # le dernier message est toujours la réponse finale
    return result["messages"]#[-1].content


## Exemples d'utilisation

Testez l'agent avec différentes questions :


In [52]:
# Exemple 1 : Question nécessitant l'outil de calcul
question1 = "Combien font 2x3-5-(2/8)*5 stp ?"
reponse1 = ask(question1)
print(f"Q: {question1}")
print(f"R: {reponse1}")


Q: Combien font 2x3-5-(2/8)*5 stp ?
R: [HumanMessage(content='Combien font 2x3-5-(2/8)*5 stp ?', additional_kwargs={}, response_metadata={}, id='ff8bff3a-0864-4bf2-9461-3f2fe82840b7'), AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "Combien font 2x3-5-(2/8)*5 stp ?" Need to compute expression: 2*3 - 5 - (2/8)*5. Let\'s compute manually: 2*3=6. 2/8 = 0.25. 0.25*5 = 1.25. So 6 - 5 - 1.25 = 1 - 1.25 = -0.25. So answer -0.25. Could also use function calculer. Use it.', 'tool_calls': [{'id': 'fc_81b96b7d-c8c8-4b04-80cb-e6823f654e88', 'function': {'arguments': '{"expression":"2*3-5-(2/8)*5"}', 'name': 'calculer'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 175, 'total_tokens': 331, 'completion_time': 0.337126338, 'completion_tokens_details': {'reasoning_tokens': 118}, 'prompt_time': 0.008026551, 'prompt_tokens_details': None, 'queue_time': 0.044746422, 'total_time': 0.345152889}, 'model_name': 'openai/gpt-os

In [53]:
# Exemple 2 : Question sur Séb
question2 = "Penses-tu qu'une mouche peut atterir sur le crâne de Séb ?"
reponse2 = ask(question2)
print(f"Q: {question2}")
print(f"R: {reponse2}")


Q: Penses-tu qu'une mouche peut atterir sur le crâne de Séb ?
R: [HumanMessage(content="Penses-tu qu'une mouche peut atterir sur le crâne de Séb ?", additional_kwargs={}, response_metadata={}, id='8166f824-57ed-432f-acce-1e9f58b2c6fd'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks in French: "Do you think a fly can land on Seb\'s head?" We need to answer. Possibly we might want to reference who is Seb. There\'s a tool whois_seb that gives description of Séb. Might be relevant to answer. We could use it to get info about Séb to answer about his head. Use the function whois_seb. Then respond.', 'tool_calls': [{'id': 'fc_a87540dc-837e-481e-8000-7b9fb934e6a4', 'function': {'arguments': '{"":{}}', 'name': 'whois_seb'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 176, 'total_tokens': 285, 'completion_time': 0.234903887, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.007004898, 'p